# Day 8 — pandas habits to unlearn: a cheat sheet

*100 Days of Polars · Phase 1: Foundations and mental model*

- **Post:** on Medium (link added after publishing)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [1]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


Twelve pandas habits, each next to its Polars replacement, on the same Olist tables.
Run the pandas cell and the Polars cell of each pair and compare.

In [2]:
import pandas as pd

items_pl = pl.read_csv("data/raw/olist_order_items_dataset.csv", try_parse_dates=True)
orders_pl = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)
items_pd = items_pl.to_pandas()
orders_pd = orders_pl.to_pandas()

## 1. Selecting columns: `df[[...]]` → `select`

In [3]:
# pandas
items_pd[["order_id", "price"]].head(3)

,order_id,price
0,00010242fe8c5a6d1ba2dd792cb16214,58.9
1,00018f77f2f0320c557190d7a144bdd3,239.9
2,000229ec398224ef6ca0657da4fc703e,199.0


In [4]:
# polars
items_pl.select("order_id", "price").head(3)

order_id,price
str,f64
"""00010242fe8c5a6d1ba2dd792cb162…",58.9
"""00018f77f2f0320c557190d7a144bd…",239.9
"""000229ec398224ef6ca0657da4fc70…",199.0


## 2. Boolean masks → `filter`

In [5]:
# pandas
items_pd[(items_pd["price"] > 100) & (items_pd["freight_value"] < 20)].shape

(21023, 7)

In [6]:
# polars
items_pl.filter(pl.col("price") > 100, pl.col("freight_value") < 20).shape

(21023, 7)

## 3. Assigning columns (`df["x"] = ...`, `assign`) → `with_columns`

In [7]:
# pandas
items_pd2 = items_pd.copy()
items_pd2["total"] = items_pd2["price"] + items_pd2["freight_value"]
items_pd2[["price", "freight_value", "total"]].head(3)

,price,freight_value,total
0,58.9,13.29,72.19
1,239.9,19.93,259.83
2,199.0,17.87,216.87


In [8]:
# polars
items_pl.with_columns(total=pl.col("price") + pl.col("freight_value")).select("price", "freight_value", "total").head(3)

price,freight_value,total
f64,f64,f64
58.9,13.29,72.19
239.9,19.93,259.83
199.0,17.87,216.87


## 4. Row-wise `apply` → rewrite as an expression

`apply(f, axis=1)` calls a Python function once per row. Polars' literal equivalent is `pl.struct(...).map_elements(f)`, which is just as slow. The real replacement is the expression that does what `f` does: for an if/else that is `when/then/otherwise`, Polars' version of `np.where`.

In [9]:
# pandas
items_pd.apply(lambda r: "high" if r["price"] > 100 else "low", axis=1).value_counts()

low     72337
high    40313
Name: count, dtype: int64

In [10]:
# polars, literal translation (slow: Python per row)
items_pl.select(
    band=pl.struct("price").map_elements(lambda r: "high" if r["price"] > 100 else "low", return_dtype=pl.String)
)["band"].value_counts()

band,count
str,u32
"""low""",72337
"""high""",40313


In [11]:
# polars, the real replacement: a vectorized expression (like np.where in pandas)
items_pl.select(band=pl.when(pl.col("price") > 100).then(pl.lit("high")).otherwise(pl.lit("low")))["band"].value_counts()

band,count
str,u32
"""high""",40313
"""low""",72337


## 5. `groupby().agg()` → `group_by().agg()` with named outputs

In [12]:
# pandas
items_pd.groupby("seller_id").agg(revenue=("price", "sum"), items=("order_id", "size")).sort_values("revenue", ascending=False).head(3)

,revenue,items
seller_id,,
4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1156
53243585a1d6dc2643021fd1853d8905,222776.05,410
4a3ca9315b744ce9f8e9374361493884,200472.92,1987


In [13]:
# polars
items_pl.group_by("seller_id").agg(revenue=pl.col("price").sum(), items=pl.len()).sort("revenue", descending=True).head(3)

seller_id,revenue,items
str,f64,u32
"""4869f7a5dfa277a7dca6462dcf3b52…",229472.63,1156
"""53243585a1d6dc2643021fd1853d89…",222776.05,410
"""4a3ca9315b744ce9f8e93743614938…",200472.92,1987


## 6. `merge` → `join`

In [14]:
# pandas
items_pd.merge(orders_pd[["order_id", "order_status"]], on="order_id", how="left").shape

(112650, 8)

In [15]:
# polars
items_pl.join(orders_pl.select("order_id", "order_status"), on="order_id", how="left").shape

(112650, 8)

## 7. `sort_values` → `sort`

In [16]:
# pandas
items_pd.sort_values(["seller_id", "price"], ascending=[True, False]).head(3)[["seller_id", "price"]]

,seller_id,price
55943,0015a82c2db000af6aaaf3ae2ecb0532,895.0
69082,0015a82c2db000af6aaaf3ae2ecb0532,895.0
93696,0015a82c2db000af6aaaf3ae2ecb0532,895.0


In [17]:
# polars
items_pl.sort("seller_id", "price", descending=[False, True]).select("seller_id", "price").head(3)

seller_id,price
str,f64
"""0015a82c2db000af6aaaf3ae2ecb05…",895.0
"""0015a82c2db000af6aaaf3ae2ecb05…",895.0
"""0015a82c2db000af6aaaf3ae2ecb05…",895.0


## 8. `fillna` → `fill_null`

In [18]:
# pandas
orders_pd["order_approved_at"].fillna(orders_pd["order_purchase_timestamp"]).isna().sum()

np.int64(0)

In [19]:
# polars
orders_pl.select(pl.col("order_approved_at").fill_null(pl.col("order_purchase_timestamp")).null_count()).item()

0

## 9. `astype` → `cast`

In [20]:
# pandas
items_pd["order_item_id"].astype("int8").dtype

dtype('int8')

In [21]:
# polars
items_pl.select(pl.col("order_item_id").cast(pl.Int8)).dtypes

[Int8]

## 10. `value_counts` → `value_counts` (returns a DataFrame)

In [22]:
# pandas
orders_pd["order_status"].value_counts().head(3)

order_status
delivered    96478
shipped       1107
canceled       625
Name: count, dtype: int64

In [23]:
# polars
orders_pl["order_status"].value_counts(sort=True).head(3)

order_status,count
str,u32
"""delivered""",96478
"""shipped""",1107
"""canceled""",625


## 11. `drop_duplicates` → `unique`

In [24]:
# pandas
items_pd.drop_duplicates(subset=["order_id"]).shape

(98666, 7)

In [25]:
# polars
items_pl.unique(subset="order_id").shape

(98666, 7)

## 12. `rename(columns=...)` → `rename`

In [26]:
# pandas
items_pd.rename(columns={"price": "item_price"}).columns[:6].tolist()

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'item_price']

In [27]:
# polars
items_pl.rename({"price": "item_price"}).columns[:6]

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'item_price']

## Two more habits: `pl.col` vs `df["col"]`, and `select` vs `with_columns`

- `pl.col("x")` is an **instruction** that runs inside a context, works lazily and refers to whatever table reaches that step of a chain. `df["x"]` is the **data itself**, taken out of `df` right now (eager only). Use `pl.col` to describe work, `df["x"]` to take results out.
- `select` returns **only** the listed columns; `with_columns` returns **all** columns plus the new ones. Reassigning after `select` drops everything you did not list.

In [28]:
band = pl.when(pl.col("price") > 100).then(pl.lit("high")).otherwise(pl.lit("low"))

print(items_pl.select(band=band).columns)              # only 'band'
print(items_pl.with_columns(band=band).columns)        # everything + 'band'

# pl.col refers to a column created earlier in the same chain
items_pl.with_columns(band=band).filter(pl.col("band") == "high").height

['band']
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'band']


40313

## How much does `apply` cost?

In [29]:
import time
import numpy as np


def timed(fn):
    t0 = time.perf_counter()
    fn()
    return (time.perf_counter() - t0) * 1000


timings = {
    "pandas apply (row-wise)": timed(
        lambda: items_pd.apply(lambda r: "high" if r["price"] > 100 else "low", axis=1)
    ),
    "pandas np.where (vectorized)": timed(lambda: np.where(items_pd["price"] > 100, "high", "low")),
    "polars map_elements (row-wise)": timed(
        lambda: items_pl.select(
            pl.struct("price").map_elements(lambda r: "high" if r["price"] > 100 else "low", return_dtype=pl.String)
        )
    ),
    "polars when/then (vectorized)": timed(
        lambda: items_pl.select(pl.when(pl.col("price") > 100).then(pl.lit("high")).otherwise(pl.lit("low")))
    ),
}
for name, ms in timings.items():
    print(f"{name:<32} {ms:8.1f} ms")

pandas apply (row-wise)             460.0 ms
pandas np.where (vectorized)          2.9 ms
polars map_elements (row-wise)       26.7 ms
polars when/then (vectorized)         0.8 ms


## Habits with no Polars equivalent (on purpose)

| pandas habit | In Polars |
|---|---|
| `inplace=True` | Not there. Every method returns a new frame; reassign it. Copies are cheap because data is shared. |
| `set_index` / `reset_index` / `.loc[label]` | No index. Keep keys as columns; use `filter` or `join`. |
| `df.iloc[5:10]` | `df.slice(5, 5)` or `df[5:10]` |
| Chained assignment `df[df.x > 1]["y"] = 0` | `df.with_columns(pl.when(pl.col("x") > 1).then(0).otherwise(pl.col("y")).alias("y"))` |
| `df.melt(...)` | `df.unpivot(...)` |
| `df.pivot_table(...)` | `df.pivot(on=..., index=..., values=..., aggregate_function=...)` |
| `object` columns of mixed types | Not allowed: one dtype per column |

## Try it yourself

1. Add a pandas vectorized version of habit 4 with `np.where` and time it too. How close does it get to Polars?
2. Pick one pandas script you wrote at work and translate three lines using this sheet.